In [2]:
!pip install scapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.2 MB/s eta 0:00:00


In [ ]:
# --- Delta-Prime (δ') FFT/Spectral Feature Extraction (v2) ---
# This script extracts Frequency Domain features SEPARATELY for
# Client-to-Server (C2S) and Server-to-Client (S2C) directions.
# This aligns with the architecture of the Beta component.

import os
import numpy as np
import pandas as pd
from scipy.fft import fft
from scapy.all import rdpcap, IP
from joblib import Parallel, delayed

# --- CONFIG ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/"
SOURCE_DIR = os.path.join(BASE_PATH, "Notebook/VPNOnlyDataset")
# Note the 'v2' in the filename
OUTPUT_CSV = os.path.join(BASE_PATH, "new approach v1/VPNOnly-fft_component_v2.csv")

# Increased to 100 to capture longer rhythmic cycles (like video buffering)
MAX_PACKETS = 100

def calculate_fft_features(sequence, prefix):
    # Returns 0s if sequence is too short
    if len(sequence) < 2:
        return {
            f'{prefix}_fft_mean': 0.0,
            f'{prefix}_fft_std': 0.0,
            f'{prefix}_fft_max': 0.0,
            f'{prefix}_fft_entropy': 0.0
        }

    # 1. Apply FFT
    # Take Magnitude (abs)
    fft_vals = np.abs(fft(sequence))

    # 2. Drop DC component (index 0) and take first half
    # (FFT is symmetric, we only need the positive frequencies)
    n = len(fft_vals)
    fft_vals = fft_vals[1 : n//2]

    if len(fft_vals) == 0:
        return {
            f'{prefix}_fft_mean': 0.0,
            f'{prefix}_fft_std': 0.0,
            f'{prefix}_fft_max': 0.0,
            f'{prefix}_fft_entropy': 0.0
        }

    # 3. Extract Statistical Features from the Spectrum
    feat_mean = float(np.mean(fft_vals))
    feat_std = float(np.std(fft_vals))
    feat_max = float(np.max(fft_vals))

    # 4. Spectral Entropy (Complexity)
    # Normalize to get a Probability Mass Function (PMF)
    psd_norm = fft_vals / (np.sum(fft_vals) + 1e-9)
    feat_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-9))

    return {
        f'{prefix}_fft_mean': feat_mean,
        f'{prefix}_fft_std': feat_std,
        f'{prefix}_fft_max': feat_max,
        f'{prefix}_fft_entropy': float(feat_entropy)
    }

def process_pcap_fft(filename):
    filepath = os.path.join(SOURCE_DIR, filename)

    try:
        # Read packets
        packets = rdpcap(filepath, count=MAX_PACKETS)

        # 1. Identify Client IP (First packet src)
        client_ip = None
        for pkt in packets:
            if IP in pkt:
                client_ip = pkt[IP].src
                break

        if client_ip is None:
            return None

        # 2. Split Directions
        c2s_sizes = []
        c2s_times = []
        s2c_sizes = []
        s2c_times = []

        for pkt in packets:
            if IP in pkt:
                size = float(pkt[IP].len)
                time = float(pkt.time)

                if pkt[IP].src == client_ip:
                    # Client -> Server
                    c2s_sizes.append(size)
                    c2s_times.append(time)
                elif pkt[IP].dst == client_ip:
                    # Server -> Client
                    s2c_sizes.append(size)
                    s2c_times.append(time)

        # 3. Calculate IATs (Inter-Arrival Times)
        # Check if lists are empty before diffing
        c2s_iats = np.diff(c2s_times).tolist() if len(c2s_times) > 1 else []
        s2c_iats = np.diff(s2c_times).tolist() if len(s2c_times) > 1 else []

        # 4. Generate Features
        features = {'filename': filename}

        # FFT on Packet Sizes (e.g., Block size consistency)
        features.update(calculate_fft_features(c2s_sizes, 'c2s_size'))
        features.update(calculate_fft_features(s2c_sizes, 's2c_size'))

        # FFT on IATs (e.g., Jitter rhythm / Buffering pauses)
        features.update(calculate_fft_features(c2s_iats, 'c2s_iat'))
        features.update(calculate_fft_features(s2c_iats, 's2c_iat'))

        return features

    except Exception as e:
        # print(f"Error: {e}")
        return None

def main():
    print(f"--- Extracting Bi-Directional FFT Features from {SOURCE_DIR} ---")
    if not os.path.exists(SOURCE_DIR):
        print("Error: Source directory not found.")
        return

    filenames = [f for f in os.listdir(SOURCE_DIR) if f.endswith('.pcap')]
    print(f"Found {len(filenames)} files.")

    # Run in Parallel
    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(process_pcap_fft)(f) for f in filenames
    )

    valid_results = [r for r in results if r is not None]
    df = pd.DataFrame(valid_results)

    print(f"Extracted FFT features for {len(df)} files.")

    # Save
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved v2 (Bi-Directional) to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

--- Extracting Bi-Directional FFT Features from /content/drive/MyDrive/1 Skripsi/Notebook/VPNOnlyDataset ---
Found 2730 files.


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 188 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 1628 tasks      | elapsed:  1.4min


Extracted FFT features for 2730 files.
Saved v2 (Bi-Directional) to /content/drive/MyDrive/1 Skripsi/new approach v1/VPNOnly-fft_component_v2.csv


[Parallel(n_jobs=-1)]: Done 2730 out of 2730 | elapsed:  1.7min finished


In [4]:
# --- Delta-Prime (δ') FFT/Spectral Feature Extraction (v2) ---
# This script extracts Frequency Domain features SEPARATELY for
# Client-to-Server (C2S) and Server-to-Client (S2C) directions.
# This aligns with the architecture of the Beta component.

import os
import numpy as np
import pandas as pd
from scipy.fft import fft
from scapy.all import rdpcap, IP
from joblib import Parallel, delayed

# --- CONFIG ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/"
# Modified: SOURCE_DIRS is now a list of directories
SOURCE_DIRS = [
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final"),
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final 2"),
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final 3"),
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final 4"),
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final 5"),
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final 6"),
    os.path.join(BASE_PATH, "Dataset/FULL DATA NON and VPN/cleaned_flows_final 7")
]
# Note the 'v2' in the filename
OUTPUT_CSV = os.path.join(BASE_PATH, "new approach v2/fft_component_v2.csv")

# Increased to 100 to capture longer rhythmic cycles (like video buffering)
MAX_PACKETS = 100

def calculate_fft_features(sequence, prefix):
    # Returns 0s if sequence is too short
    if len(sequence) < 2:
        return {
            f'{prefix}_fft_mean': 0.0,
            f'{prefix}_fft_std': 0.0,
            f'{prefix}_fft_max': 0.0,
            f'{prefix}_fft_entropy': 0.0
        }

    # 1. Apply FFT
    # Take Magnitude (abs)
    fft_vals = np.abs(fft(sequence))

    # 2. Drop DC component (index 0) and take first half
    # (FFT is symmetric, we only need the positive frequencies)
    n = len(fft_vals)
    fft_vals = fft_vals[1 : n//2]

    if len(fft_vals) == 0:
        return {
            f'{prefix}_fft_mean': 0.0,
            f'{prefix}_fft_std': 0.0,
            f'{prefix}_fft_max': 0.0,
            f'{prefix}_fft_entropy': 0.0
        }

    # 3. Extract Statistical Features from the Spectrum
    feat_mean = float(np.mean(fft_vals))
    feat_std = float(np.std(fft_vals))
    feat_max = float(np.max(fft_vals))

    # 4. Spectral Entropy (Complexity)
    # Normalize to get a Probability Mass Function (PMF)
    psd_norm = fft_vals / (np.sum(fft_vals) + 1e-9)
    feat_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-9))

    return {
        f'{prefix}_fft_mean': feat_mean,
        f'{prefix}_fft_std': feat_std,
        f'{prefix}_fft_max': feat_max,
        f'{prefix}_fft_entropy': float(feat_entropy)
    }

def process_pcap_fft(filepath):
    try:
        # Read packets
        packets = rdpcap(filepath, count=MAX_PACKETS)
        filename = os.path.basename(filepath)

        # 1. Identify Client IP (First packet src)
        client_ip = None
        for pkt in packets:
            if IP in pkt:
                client_ip = pkt[IP].src
                break

        if client_ip is None:
            return None

        # 2. Split Directions
        c2s_sizes = []
        c2s_times = []
        s2c_sizes = []
        s2c_times = []

        for pkt in packets:
            if IP in pkt:
                size = float(pkt[IP].len)
                time = float(pkt.time)

                if pkt[IP].src == client_ip:
                    # Client -> Server
                    c2s_sizes.append(size)
                    c2s_times.append(time)
                elif pkt[IP].dst == client_ip:
                    # Server -> Client
                    s2c_sizes.append(size)
                    s2c_times.append(time)

        # 3. Calculate IATs (Inter-Arrival Times)
        # Check if lists are empty before diffing
        c2s_iats = np.diff(c2s_times).tolist() if len(c2s_times) > 1 else []
        s2c_iats = np.diff(s2c_times).tolist() if len(s2c_times) > 1 else []

        # 4. Generate Features
        features = {'filename': filename}

        # FFT on Packet Sizes (e.g., Block size consistency)
        features.update(calculate_fft_features(c2s_sizes, 'c2s_size'))
        features.update(calculate_fft_features(s2c_sizes, 's2c_size'))

        # FFT on IATs (e.g., Jitter rhythm / Buffering pauses)
        features.update(calculate_fft_features(c2s_iats, 'c2s_iat'))
        features.update(calculate_fft_features(s2c_iats, 's2c_iat'))

        return features

    except Exception as e:
        # print(f"Error processing {filepath}: {e}")
        return None

def main():
    print(f"--- Extracting Bi-Directional FFT Features from specified directories ---")
    all_filepaths = []
    for source_dir in SOURCE_DIRS:
        if not os.path.exists(source_dir):
            print(f"Warning: Source directory not found: {source_dir}")
            continue

        for root, _, files in os.walk(source_dir):
            for f in files:
                if f.endswith('.pcap'):
                    all_filepaths.append(os.path.join(root, f))

    print(f"Found {len(all_filepaths)} files across all specified directories.")

    # Run in Parallel
    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(process_pcap_fft)(fp) for fp in all_filepaths
    )

    valid_results = [r for r in results if r is not None]
    df = pd.DataFrame(valid_results)

    print(f"Extracted FFT features for {len(df)} files.")

    # Save
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved v2 (Bi-Directional) to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

--- Extracting Bi-Directional FFT Features from specified directories ---
Found 13487 files across all specified directories.


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done  68 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done 158 tasks      | elapsed:  5.2min
[Parallel(n_jobs=-1)]: Done 1070 tasks      | elapsed:  5.3min
[Parallel(n_jobs=-1)]: Done 3302 tasks      | elapsed:  5.8min
[Parallel(n_jobs=-1)]: Done 6470 tasks      | elapsed:  6.2min
[Parallel(n_jobs=-1)]: Done 7068 tasks      | elapsed:  6.9min
[Parallel(n_jobs=-1)]: Done 7338 tasks      | elapsed:  7.4min
[Parallel(n_jobs=-1)]: Done 7644 tasks      | elapsed:  8.2min
[Parallel(n_jobs=-1)]: Done 7986 tasks      | elapsed:  9.0min
[Parallel(n_jobs=-1)]: Done 8364 tasks      | elapsed:  9.8min
[Parallel(n_jobs=-1)]: Done 8778 tasks      | elapsed: 10.8min
[Parallel(n_jobs=-1)]: Done 9228 tasks      | elapsed: 11.8min
[Parallel(n_jobs=-1)]: Done 9714 tasks      | elapsed: 12.9min
[Parallel(n_jobs=-1)]: Done 10236 tasks      

Extracted FFT features for 13305 files.
Saved v2 (Bi-Directional) to /content/drive/MyDrive/1 Skripsi/new approach v2/fft_component_v2.csv
